## Step 3 — extract and process major roads, pt 2
**# of cells in notebook:** 2

**Purpose:** at this point, the roads geodatabase should contain `roads_8`. The question now is whether there are 'spatial artifacts' generated by the preceding roads processing scripts. If they exist, they are typically narrow polygons from divided roads that did not collapse. In the scripts below, we perform compactness tests on the polygons created by `roads_8` lines to determine whether there exist aftifcats that need to be skeletonized.  

**Input:**

- a geodatabase with:  `roads_8` layer.

**Output:**

- final output is `roads_9`. However, the script outputs each intermediate processing layer, so that it may be inspected for QA purposes. 

**Main logic:**

1.	Polygonize `roads_8`
2.	For each polygon feature, calculate the radius of the equal area circle (EAC).  
3.	Draw that circle around the polygon centroid. Compute the intersection area of the EAC and its associated polygon and the overlap ratio.  
4.	Features that meet both ratio and size criteria are written to `roads_8_7` for further processing. 
5.	If no features meet the criteria, `roads_8` is written to `roads_9`
6.	If features meet the criteria, `roads_8_7` is written, and you must run the second cell to generate `roads_9`


In [ ]:
import arcpy
import os
import math
import time

# -------------------------------------------------------------------
# ENVIRONMENT
# -------------------------------------------------------------------
arcpy.ResetEnvironments()

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100"

# Clear any inherited tolerance settings from previous geoprocessing work
arcpy.ClearEnvironment("XYTolerance")
arcpy.ClearEnvironment("XYResolution")

# -------------------------------------------------------------------
# GEODATABASE
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
arcpy.env.workspace = gdb

# -------------------------------------------------------------------
# KNOWN FIELD NAMES FROM SUCCESSFUL RUN
# -------------------------------------------------------------------
# In roads_8_5 and roads_8_6:
# ORIG_FID comes from roads_8_3 / roads_8_4 and identifies the original roads_8_2 polygon.
# FID_roads_8_2 identifies the polygon from roads_8_2 in the PairwiseIntersect output.
ORIG_FID_FIELD = "ORIG_FID"
POLYGON_ID_FIELD = "FID_roads_8_2"

# -------------------------------------------------------------------
# INPUTS / OUTPUTS
# -------------------------------------------------------------------
roads_8 = os.path.join(gdb, "roads_8")

roads_8_1 = os.path.join(gdb, "roads_8_1")
roads_8_2 = os.path.join(gdb, "roads_8_2")
roads_8_3 = os.path.join(gdb, "roads_8_3")
roads_8_4 = os.path.join(gdb, "roads_8_4")
roads_8_5 = os.path.join(gdb, "roads_8_5")
roads_8_6 = os.path.join(gdb, "roads_8_6")
roads_8_7 = os.path.join(gdb, "roads_8_7")

# Output used if no candidate polygons are selected
roads_9 = os.path.join(gdb, "roads_9")

# -------------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------------
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def field_exists(fc, field_name):
    return field_name in [f.name for f in arcpy.ListFields(fc)]


def count_features(fc):
    return int(arcpy.management.GetCount(fc)[0])


def describe_fc(fc):
    if not arcpy.Exists(fc):
        print(f"Does not exist: {fc}")
        return

    d = arcpy.Describe(fc)
    sr_name = d.spatialReference.name if d.spatialReference else "Unknown"

    print(f"Dataset: {fc}")
    print(f"  Shape type: {d.shapeType}")
    print(f"  Spatial ref: {sr_name}")
    print(f"  Feature count: {count_features(fc)}")


def print_fields(fc):
    print(f"\nFields in {os.path.basename(fc)}:")
    for f in arcpy.ListFields(fc):
        print(f"  {f.name} | {f.type}")


def run_step(step_name, func, *args, **kwargs):
    print(f"\n--- Starting: {step_name} ---")
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    print(f"--- Finished: {step_name} in {elapsed:.1f} seconds ---")
    return result


def add_field_if_missing(fc, field_name, field_type):
    if not field_exists(fc, field_name):
        arcpy.management.AddField(fc, field_name, field_type)
        print(f"Added field: {field_name}")
    else:
        print(f"Field already exists: {field_name}")


def validate_required_fields(fc, required_fields):
    existing = [f.name for f in arcpy.ListFields(fc)]
    missing = [f for f in required_fields if f not in existing]

    if missing:
        print_fields(fc)
        raise ValueError(
            f"Missing required field(s) in {fc}: {missing}"
        )


# -------------------------------------------------------------------
# WORKFLOW FUNCTIONS
# -------------------------------------------------------------------
def dissolve_roads():
    """
    Step 1:
    Dissolve roads_8 to roads_8_1.
    """
    delete_if_exists(roads_8_1)

    arcpy.analysis.PairwiseDissolve(
        in_features=roads_8,
        out_feature_class=roads_8_1,
        dissolve_field=None,
        multi_part="SINGLE_PART"
    )

    print(f"PairwiseDissolve output: {roads_8_1}")
    describe_fc(roads_8_1)


def feature_to_polygon():
    """
    Step 2:
    Convert dissolved road lines to polygons.
    """
    delete_if_exists(roads_8_2)

    arcpy.management.FeatureToPolygon(
        in_features=roads_8_1,
        out_feature_class=roads_8_2,
        cluster_tolerance=None,
        attributes="NO_ATTRIBUTES"
    )

    print(f"FeatureToPolygon output: {roads_8_2}")
    describe_fc(roads_8_2)
    print_fields(roads_8_2)


def add_and_calculate_fields_roads_8_2():
    """
    Step 3:
    Add area_m2 and radius_m to roads_8_2.

    Do not use a custom field called shape_area in a file geodatabase.
    ArcGIS already has a managed Shape_Area field.
    """
    add_field_if_missing(roads_8_2, "area_m2", "DOUBLE")

    arcpy.management.CalculateGeometryAttributes(
        roads_8_2,
        [["area_m2", "AREA"]],
        area_unit="SQUARE_METERS"
    )

    add_field_if_missing(roads_8_2, "radius_m", "DOUBLE")

    arcpy.management.CalculateField(
        roads_8_2,
        "radius_m",
        "math.sqrt(!area_m2! / math.pi)",
        "PYTHON3",
        code_block="import math"
    )

    print("Calculated area_m2 and radius_m on roads_8_2.")
    print_fields(roads_8_2)


def feature_to_point():
    """
    Step 4:
    Convert polygons to centroid points.

    FeatureToPoint creates ORIG_FID, which records the source polygon OBJECTID
    from roads_8_2.
    """
    delete_if_exists(roads_8_3)

    arcpy.management.FeatureToPoint(
        in_features=roads_8_2,
        out_feature_class=roads_8_3,
        point_location="CENTROID"
    )

    print(f"FeatureToPoint output: {roads_8_3}")
    describe_fc(roads_8_3)
    print_fields(roads_8_3)

    validate_required_fields(roads_8_3, [ORIG_FID_FIELD])


def pairwise_buffer():
    """
    Step 5:
    Buffer each centroid using radius_m.
    """
    delete_if_exists(roads_8_4)

    arcpy.analysis.PairwiseBuffer(
        in_features=roads_8_3,
        out_feature_class=roads_8_4,
        buffer_distance_or_field="radius_m",
        dissolve_option="NONE",
        method="PLANAR"
    )

    print(f"PairwiseBuffer output: {roads_8_4}")
    describe_fc(roads_8_4)
    print_fields(roads_8_4)

    validate_required_fields(roads_8_4, [ORIG_FID_FIELD, "area_m2", "radius_m"])


def pairwise_intersect():
    """
    Step 6:
    Intersect the centroid buffers with the original polygons.

    Output roads_8_5 contains:
      ORIG_FID        = original polygon ID carried from the centroid/buffer
      FID_roads_8_2  = polygon ID from roads_8_2 in the intersect
    """
    delete_if_exists(roads_8_5)

    arcpy.analysis.PairwiseIntersect(
        in_features=[roads_8_4, roads_8_2],
        out_feature_class=roads_8_5
    )

    print(f"PairwiseIntersect output: {roads_8_5}")
    describe_fc(roads_8_5)
    print_fields(roads_8_5)

    validate_required_fields(
        roads_8_5,
        [ORIG_FID_FIELD, POLYGON_ID_FIELD, "area_m2", "radius_m"]
    )


def select_and_export_roads_8_6():
    """
    Step 7:
    Select intersections where each buffer is intersecting its own source polygon.

    Correct SQL from successful run:
        ORIG_FID = FID_roads_8_2
    """
    delete_if_exists(roads_8_6)
    delete_if_exists("roads_8_5_layer")

    validate_required_fields(roads_8_5, [ORIG_FID_FIELD, POLYGON_ID_FIELD])

    sql_query = (
        f"{arcpy.AddFieldDelimiters(gdb, ORIG_FID_FIELD)} = "
        f"{arcpy.AddFieldDelimiters(gdb, POLYGON_ID_FIELD)}"
    )

    print("\nSelection SQL for roads_8_6:")
    print(sql_query)

    arcpy.management.MakeFeatureLayer(
        roads_8_5,
        "roads_8_5_layer",
        sql_query
    )

    selected_count = count_features("roads_8_5_layer")
    print(f"Selected rows for roads_8_6: {selected_count}")

    arcpy.management.CopyFeatures("roads_8_5_layer", roads_8_6)
    arcpy.management.Delete("roads_8_5_layer")

    print(f"Selected rows exported to: {roads_8_6}")
    describe_fc(roads_8_6)
    print_fields(roads_8_6)

    validate_required_fields(
        roads_8_6,
        [ORIG_FID_FIELD, POLYGON_ID_FIELD, "area_m2", "radius_m"]
    )


def add_fields_and_calculate_roads_8_6():
    """
    Step 8:
    Calculate the intersection area and the ratio:
        ratio = area_intsc / area_m2
    """
    add_field_if_missing(roads_8_6, "area_intsc", "DOUBLE")

    arcpy.management.CalculateGeometryAttributes(
        roads_8_6,
        [["area_intsc", "AREA"]],
        area_unit="SQUARE_METERS"
    )

    add_field_if_missing(roads_8_6, "ratio", "DOUBLE")

    arcpy.management.CalculateField(
        roads_8_6,
        "ratio",
        "!area_intsc! / !area_m2! if !area_m2! not in (None, 0) else None",
        "PYTHON3"
    )

    print("Calculated area_intsc and ratio on roads_8_6.")
    print_fields(roads_8_6)

    validate_required_fields(
        roads_8_6,
        [POLYGON_ID_FIELD, "area_m2", "area_intsc", "ratio"]
    )


def identify_and_export_roads_8_7():
    """
    Step 9:
    Select records from roads_8_6 where:
        area_m2 <= 275000
        ratio <= 0.375

    Then use FID_roads_8_2 to select source polygons from roads_8_2
    and export them to roads_8_7.

    If no records are selected, copy roads_8 to roads_9.
    """
    delete_if_exists(roads_8_7)
    delete_if_exists("roads_8_6_layer")
    delete_if_exists("roads_8_2_layer")

    validate_required_fields(
        roads_8_6,
        [POLYGON_ID_FIELD, "area_m2", "ratio"]
    )

    query = (
        f"{arcpy.AddFieldDelimiters(gdb, 'area_m2')} <= 275000 "
        f"AND {arcpy.AddFieldDelimiters(gdb, 'ratio')} <= 0.375"
    )

    print("\nSelection query for candidate polygons:")
    print(query)

    arcpy.management.MakeFeatureLayer(
        roads_8_6,
        "roads_8_6_layer",
        query
    )

    selected_count = count_features("roads_8_6_layer")
    print(f"Candidate records selected from roads_8_6: {selected_count}")

    if selected_count == 0:
        delete_if_exists(roads_9)
        arcpy.management.CopyFeatures(roads_8, roads_9)
        print(f"No features selected. Copied roads_8 to roads_9: {roads_9}")
        arcpy.management.Delete("roads_8_6_layer")
        return

    polygon_ids = []
    with arcpy.da.SearchCursor("roads_8_6_layer", [POLYGON_ID_FIELD]) as cursor:
        for row in cursor:
            if row[0] is not None:
                polygon_ids.append(int(row[0]))

    polygon_ids = sorted(set(polygon_ids))
    print(f"Unique source polygon IDs to select from roads_8_2: {len(polygon_ids)}")
    print(f"Polygon IDs: {polygon_ids}")

    if not polygon_ids:
        delete_if_exists(roads_9)
        arcpy.management.CopyFeatures(roads_8, roads_9)
        print(f"No polygon IDs found. Copied roads_8 to roads_9: {roads_9}")
        arcpy.management.Delete("roads_8_6_layer")
        return

    roads_8_2_oid = arcpy.Describe(roads_8_2).OIDFieldName
    oid_delimited = arcpy.AddFieldDelimiters(gdb, roads_8_2_oid)

    fid_query = f"{oid_delimited} IN ({','.join(map(str, polygon_ids))})"

    print("\nSelection SQL for roads_8_7:")
    print(fid_query)

    arcpy.management.MakeFeatureLayer(
        roads_8_2,
        "roads_8_2_layer",
        fid_query
    )

    selected_poly_count = count_features("roads_8_2_layer")
    print(f"Polygons selected from roads_8_2: {selected_poly_count}")

    arcpy.management.CopyFeatures("roads_8_2_layer", roads_8_7)

    arcpy.management.Delete("roads_8_6_layer")
    arcpy.management.Delete("roads_8_2_layer")

    print(f"roads_8_7 output written to: {roads_8_7}")
    describe_fc(roads_8_7)
    print_fields(roads_8_7)


# -------------------------------------------------------------------
# RUN
# -------------------------------------------------------------------
start_time = time.time()

print("Starting roads_8 post-processing workflow...")
print(f"Workspace: {gdb}")

if not arcpy.Exists(roads_8):
    raise ValueError(f"Required input does not exist: {roads_8}")

describe_fc(roads_8)

run_step("1. Pairwise dissolve roads_8 to roads_8_1", dissolve_roads)
run_step("2. FeatureToPolygon roads_8_1 to roads_8_2", feature_to_polygon)
run_step("3. Add area_m2 and radius_m to roads_8_2", add_and_calculate_fields_roads_8_2)
run_step("4. FeatureToPoint roads_8_2 to roads_8_3", feature_to_point)
run_step("5. PairwiseBuffer roads_8_3 to roads_8_4", pairwise_buffer)
run_step("6. PairwiseIntersect roads_8_4 and roads_8_2 to roads_8_5", pairwise_intersect)
run_step("7. Select self-matching intersections to roads_8_6", select_and_export_roads_8_6)
run_step("8. Add area_intsc and ratio to roads_8_6", add_fields_and_calculate_roads_8_6)
run_step("9. Identify and export roads_8_7", identify_and_export_roads_8_7)

elapsed = time.time() - start_time
print(f"\nProcessing completed in {elapsed / 60:.2f} minutes.")

In [ ]:
import arcpy
import os
import time

# -------------------------------------------------------------------
# ENVIRONMENT
# -------------------------------------------------------------------
arcpy.ResetEnvironments()

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100"

# Avoid inherited XY tolerance / resolution settings from prior runs
arcpy.ClearEnvironment("XYTolerance")
arcpy.ClearEnvironment("XYResolution")

# -------------------------------------------------------------------
# GEODATABASE
# -------------------------------------------------------------------
gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
arcpy.env.workspace = gdb

# -------------------------------------------------------------------
# INPUTS / OUTPUTS
# -------------------------------------------------------------------
roads_8_1 = os.path.join(gdb, "roads_8_1")
roads_8_7 = os.path.join(gdb, "roads_8_7")
roads_8_8 = os.path.join(gdb, "roads_8_8")
roads_8_9 = os.path.join(gdb, "roads_8_9")
roads_9 = os.path.join(gdb, "roads_9")

# -------------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------------
def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def count_features(fc):
    return int(arcpy.management.GetCount(fc)[0])


def describe_fc(fc):
    if not arcpy.Exists(fc):
        print(f"Does not exist: {fc}")
        return

    d = arcpy.Describe(fc)
    sr_name = d.spatialReference.name if d.spatialReference else "Unknown"

    print(f"Dataset: {fc}")
    print(f"  Shape type: {d.shapeType}")
    print(f"  Spatial ref: {sr_name}")
    print(f"  Feature count: {count_features(fc)}")


def run_step(step_name, func, *args, **kwargs):
    print(f"\n--- Starting: {step_name} ---")
    t0 = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - t0
    print(f"--- Finished: {step_name} in {elapsed:.1f} seconds ---")
    return result


def require_exists(path, label):
    if not arcpy.Exists(path):
        raise ValueError(f"Required {label} does not exist: {path}")


# -------------------------------------------------------------------
# WORKFLOW FUNCTIONS
# -------------------------------------------------------------------
def erase_features():
    """
    Step 1:
    Erase roads_8_1 using roads_8_7.
    
    Output:
        roads_8_8
    """
    require_exists(roads_8_1, "input roads_8_1")
    require_exists(roads_8_7, "erase feature roads_8_7")

    delete_if_exists(roads_8_8)

    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    arcpy.analysis.Erase(
        in_features=roads_8_1,
        erase_features=roads_8_7,
        out_feature_class=roads_8_8
    )

    print(f"Erase output written to: {roads_8_8}")
    describe_fc(roads_8_8)


def collapse_hydro_polygons():
    """
    Step 2:
    Collapse roads_8_7 polygons to line features, using roads_8_8
    as connecting features.
    
    Output:
        roads_8_9
    """
    require_exists(roads_8_7, "input roads_8_7")
    require_exists(roads_8_8, "connecting features roads_8_8")

    delete_if_exists(roads_8_9)

    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    arcpy.cartography.CollapseHydroPolygon(
        in_features=roads_8_7,
        out_line_feature_class=roads_8_9,
        merge_adjacent_input_polygons="MERGE_ADJACENT",
        connecting_features=roads_8_8
    )

    print(f"CollapseHydroPolygon output written to: {roads_8_9}")
    describe_fc(roads_8_9)


def merge_datasets():
    """
    Step 3:
    Merge roads_8_8 and roads_8_9 into roads_9.
    
    Output:
        roads_9
    """
    require_exists(roads_8_8, "input roads_8_8")
    require_exists(roads_8_9, "input roads_8_9")

    delete_if_exists(roads_9)

    arcpy.ClearEnvironment("XYTolerance")
    arcpy.ClearEnvironment("XYResolution")

    arcpy.management.Merge(
        inputs=[roads_8_8, roads_8_9],
        output=roads_9
    )

    print(f"Merge output written to: {roads_9}")
    describe_fc(roads_9)


# -------------------------------------------------------------------
# RUN
# -------------------------------------------------------------------
start_time = time.time()

print("Starting roads_8_7 to roads_9 workflow...")
print(f"Workspace: {gdb}")

# This script should only be run when roads_8_7 exists.
# If the prior script selected zero candidate polygons, it already copied roads_8 to roads_9,
# and this script is not needed.
if not arcpy.Exists(roads_8_7):
    raise ValueError(
        f"roads_8_7 does not exist: {roads_8_7}\n"
        "This usually means the previous script selected zero candidate polygons. "
        "In that case, roads_9 should already have been created by copying roads_8."
    )

describe_fc(roads_8_1)
describe_fc(roads_8_7)

run_step("1. Erase roads_8_1 by roads_8_7 to create roads_8_8", erase_features)
run_step("2. Collapse roads_8_7 to roads_8_9 using roads_8_8", collapse_hydro_polygons)
run_step("3. Merge roads_8_8 and roads_8_9 to create roads_9", merge_datasets)

elapsed = time.time() - start_time
print(f"\nProcessing completed in {elapsed / 60:.2f} minutes.")